# Chapter 1 lab — Why a robot can't wait for a slow brain

Companion to **[Chapter 1 — LLM AI vs Physical AI](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/01-llm-vs-physical-ai/)**.

Chapter 1 claims Physical AI must act *every 10–20 milliseconds, forever*, and that a single late command is irreversible — *physics already happened*. This notebook lets you **feel** that claim instead of taking it on faith.

We balance an inverted pendulum (a broom on your palm) in MuJoCo, then run the **same controller** at three decision rates and watch where it falls over.

~5 minutes. **CPU is fine** — no GPU needed for this one.

## Step 1 — install MuJoCo (~1 min)

In [ ]:
!pip install -q mujoco mediapy
print("install OK")

## Step 2 — a pendulum that must be actively balanced

A single hinge with a heavy tip. Gravity tips it over; a motor at the hinge can push it back. Left alone it falls — just like our hexapod face-plants without active control.

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")  # headless offscreen GL for Colab
import mujoco, mediapy as media, numpy as np

XML = """
<mujoco>
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1" rgba=".7 .7 .7 1"/>
    <body pos="0 0 0.6" euler="0 8 0">       <!-- start tilted 8 degrees -->
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0  0 0 0.5" size="0.03" rgba=".2 .4 .8 1"/>
      <geom type="sphere" pos="0 0 0.5" size="0.07" rgba=".8 .2 .2 1" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <motor joint="hinge" ctrlrange="-20 20"/>  <!-- the 'servo' -->
  </actuator>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(XML)
print("joints:", model.njnt, "  actuators:", model.nu, "  timestep:", model.opt.timestep, "s")

## Step 3 — one simple controller, three decision rates

The controller is a textbook PD law: *push back proportional to how far it's tilted and how fast it's tipping.* The **only** thing we change between runs is `control_period_ms` — how often the controller is allowed to look and react. Between decisions the last command is held, exactly like a real robot waiting on a slow brain.

- **20 ms** — a fast Physical-AI control loop (50 Hz).
- **200 ms** — sluggish.
- **1000 ms** — “LLM speed”: one thoughtful decision per second.

In [ ]:
def run(control_period_ms, seconds=4.0):
    data = mujoco.MjData(model)
    data.qpos[0] = np.deg2rad(8)            # same 8-degree initial tilt every time
    renderer = mujoco.Renderer(model, height=240, width=320)
    steps = int(seconds / model.opt.timestep)
    decide_every = max(1, int((control_period_ms/1000) / model.opt.timestep))
    Kp, Kd = 80.0, 12.0                     # PD gains (same for all runs)
    ctrl = 0.0
    frames, max_tilt = [], 0.0
    for t in range(steps):
        if t % decide_every == 0:           # controller only reacts this often
            angle, rate = data.qpos[0], data.qvel[0]
            ctrl = np.clip(-Kp*angle - Kd*rate, -20, 20)
        data.ctrl[0] = ctrl                 # held between decisions
        mujoco.mj_step(model, data)
        max_tilt = max(max_tilt, abs(np.rad2deg(data.qpos[0])))
        if t % 10 == 0:
            renderer.update_scene(data)
            frames.append(renderer.render())
    fell = max_tilt > 80
    return frames, max_tilt, fell

for ms in (20, 200, 1000):
    _, max_tilt, fell = run(ms)
    verdict = "FELL OVER" if fell else "stayed up"
    print(f"decide every {ms:>4} ms  ->  max tilt {max_tilt:5.1f} deg  ->  {verdict}")

## Step 4 — watch the fast loop vs the slow loop

In [ ]:
fast_frames, _, _ = run(20)
slow_frames, _, _ = run(1000)
print("LEFT idea: 20 ms loop (fast).   RIGHT idea: 1000 ms loop (LLM speed).")
print("--- 20 ms control loop ---")
media.show_video(fast_frames, fps=30)
print("--- 1000 ms control loop ---")
media.show_video(slow_frames, fps=30)

!!! abstract "What you just proved"
    - The **controller never changed** — only how often it was allowed to act.
    - At 20 ms it balances; at 1000 ms the pendulum is already on the floor before the next decision. *Physics already happened.*
    - This is why you can't “put ChatGPT in the control loop”: not because it isn't smart, but because balance is a deadline, and the deadline is milliseconds.

**Try it:** lower `Kp`/`Kd` and see the fast loop fail too — good gains *and* a fast loop are both required. Next: **[Chapter 2 — Legs and Fingers](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/02-legs-and-fingers/)**.